# Day 4 — SQL: Aggregation
**Topics:** SUM, COUNT, AVG, MIN, MAX, GROUP BY, HAVING

In [ ]:
%load_ext sql
USERNAME = 'postgres'
PASSWORD = 'hariom'
HOST     = 'localhost'
PORT     = '5432'
DBNAME   = 'postgres'
%sql postgresql://{USERNAME}:{PASSWORD}@{HOST}:{PORT}/{DBNAME}

## Setup — create and populate the sales table

In [ ]:
%%sql
-- ============================================================
-- 1. Exploration table: sales
-- ============================================================
DROP TABLE IF EXISTS sales;

CREATE TABLE sales (
    sale_id    VARCHAR(10) PRIMARY KEY,
    product_id VARCHAR(10),
    region     VARCHAR(20),
    sale_date  DATE,
    quantity   INT,
    unit_price NUMERIC(10,2)
);

INSERT INTO sales VALUES
  ('S001','P001','North','2024-01-05', 10, 29.99),
  ('S002','P002','South','2024-01-07',  5, 49.99),
  ('S003','P001','East', '2024-01-10', 20, 29.99),
  ('S004','P003','West', '2024-01-12', 15, 99.99),
  ('S005','P002','North','2024-01-15',  8, 49.99),
  ('S006','P001','South','2024-02-01', 30, 29.99),
  ('S007','P003','East', '2024-02-03',  2, 99.99),
  ('S008','P004','West', '2024-02-08', 25, 14.99),
  ('S009','P004','North','2024-02-11', 40, 14.99),
  ('S010','P002','East', '2024-02-14', 12, 49.99),
  ('S011','P001','West', '2024-03-01', 18, 29.99),
  ('S012','P003','South','2024-03-05',  7, 99.99),
  ('S013','P004','East', '2024-03-09', 22, 14.99),
  ('S014','P002','West', '2024-03-12',  3, 49.99),
  ('S015','P001','North','2024-03-20', 14, 29.99);

-- ============================================================
-- 2. Practice Question table (pq_ prefix)
-- ============================================================
DROP TABLE IF EXISTS pq_sales;
CREATE TABLE pq_sales AS SELECT * FROM sales;

-- Verify
SELECT 'sales'    AS tbl, COUNT(*) AS rows FROM sales
UNION ALL
SELECT 'pq_sales' AS tbl, COUNT(*) AS rows FROM pq_sales;

## 1. COUNT — total rows vs non-NULL

In [ ]:
%%sql
SELECT COUNT(*)         AS total_rows,
       COUNT(region)    AS non_null_region,
       COUNT(DISTINCT region) AS distinct_regions
FROM   sales;

## 2. SUM — total quantity and total revenue

In [ ]:
%%sql
SELECT SUM(quantity)                                  AS total_units,
       ROUND(SUM(quantity * unit_price)::NUMERIC, 2)  AS total_revenue
FROM   sales;

## 3. GROUP BY — aggregation per group

In [ ]:
%%sql
-- Revenue per region — every SELECT column must be in GROUP BY or aggregated
SELECT  region,
        ROUND(SUM(quantity * unit_price)::NUMERIC, 2)  AS total_revenue,
        SUM(quantity)                                  AS total_units,
        COUNT(*)                                       AS num_sales
FROM    sales
GROUP   BY region
ORDER   BY total_revenue DESC;

## 4. MIN / MAX / AVG

In [ ]:
%%sql
SELECT  region,
        MIN(unit_price)                            AS min_price,
        MAX(unit_price)                            AS max_price,
        ROUND(AVG(unit_price)::NUMERIC, 2)         AS avg_price
FROM    sales
GROUP   BY region
ORDER   BY region;

## 5. WHERE vs HAVING — critical difference

In [ ]:
%%sql
-- WHERE filters rows BEFORE grouping (cannot reference aggregates)
-- HAVING filters groups AFTER aggregation

-- Products with total units sold > 50
SELECT  product_id,
        SUM(quantity)  AS total_units_sold
FROM    sales
GROUP   BY product_id
HAVING  SUM(quantity) > 50
ORDER   BY total_units_sold DESC;

In [ ]:
%%sql
-- WHERE + HAVING combined:
-- Only 2024-Q1 sales, then filter groups with >10 units
SELECT  product_id,
        SUM(quantity)  AS total_units
FROM    sales
WHERE   sale_date < '2024-02-01'    -- WHERE: filter rows first
GROUP   BY product_id
HAVING  SUM(quantity) > 10          -- HAVING: filter groups after
ORDER   BY total_units DESC;

## 6. All Five Aggregates in One Query

In [ ]:
%%sql
SELECT  region,
        COUNT(*)                                    AS num_transactions,
        MIN(unit_price)                             AS min_price,
        MAX(unit_price)                             AS max_price,
        ROUND(AVG(unit_price)::NUMERIC, 2)          AS avg_price,
        SUM(quantity)                               AS total_qty
FROM    sales
GROUP   BY region
ORDER   BY region;

## 7. GROUP BY multiple columns

In [ ]:
%%sql
-- Revenue per region + product
SELECT  region,
        product_id,
        ROUND(SUM(quantity * unit_price)::NUMERIC, 2)  AS revenue,
        SUM(quantity)                                  AS units
FROM    sales
GROUP   BY region, product_id
ORDER   BY region, revenue DESC;